# 📗 LLM 기반 NER: 개체를 프로그램이 쓸 수 있게

지난 시간엔 **규칙 기반 NER** 을 표준 사전으로 만들어, 미등록어(`statins`)·경계(`Warfarin-related`)·대소문자(`large`)·약어 충돌(`OTC`)에서 막히는 걸 보았습니다. 이번 시간엔 그 한계를 **LLM** 으로 넘습니다. 재학습도 사전도 없이 프롬프트로 개체를 뽑고, 그 결과를 프로그램이 바로 쓰도록 **구조화된 JSON** 으로 받은 뒤, **후처리로 다듬어** 신뢰할 수 있는 개체 목록을 만듭니다.

그리고 마지막에 새 문제를 하나 만납니다. LLM 이 **사전에 없는 이름**을 뽑아 오면 그것을 그래프에 어떻게 넣어야 할까요.

## ⏪ 복습: 지난 시간까지

- **NER**: 문장에서 개체의 **구간과 유형**을 찾는 작업. IE 4단계의 첫 단계입니다.
- **규칙 기반 NER**: 표준 사전(`name2id.json`)에 있는 낱말만 잡았습니다. 네 지점에서 막혔습니다.
- **유형 5종**: `Compound`·`Gene`·`Disease`·`Symptom`·`PharmacologicClass`. 앞 단원 지식 그래프의 노드 레이블과 같은 문자열입니다.
- **구조화된 출력**(LangChain 단원): `with_structured_output(스키마)` 로 모델 답을 **정해진 구조**로 받았습니다. 오늘 **2-1** 에서 이 스킬을 개체 추출에 그대로 다시 씁니다. 새 문법이 아닙니다.

**오늘의 목표**

**1. LLM 으로 개체 뽑기**
- [ ] (1-1) **zero-shot** 프롬프트로 개체를 뽑고, 규칙 기반이 놓친 이름이 잡히는 것을 확인한다.

**2. 구조화 출력**
- [ ] (2-1) **`with_structured_output(EntityList)`** 와 **`Literal`** 로 형식과 유형을 강제한다.

**3. 후처리**
- [ ] (3-1) 허용 **유형**에 없는 개체(`Other` 포함)를 버린다.
- [ ] (3-2) 개체 이름이 **원문에 나오는지** 확인해 환각을 걸러 내고, 그 필터가 진짜 개체를 잘못 버리는 자리도 본다.
- [ ] (3-3) 같은 `(이름, 유형)` **중복**을 없앤다.

**4. 배치 적용과 비교**
- [ ] (4-1) 논문 여러 편에 추출기를 **반복문**으로 돌려 유형 분포를 집계한다.
- [ ] (4-2) **규칙 기반과 LLM** 을 같은 발췌에 나란히 놓고, 사전에 없는 이름이 왜 문제인지 본다.
- [ ] (4-3) **few-shot** 이 형식을 유도할 뿐 보장하지 않는 것을 확인한다.

아래 준비 셀을 먼저 실행하세요. **본인 OpenAI API 키가 필요합니다**(`.env` 의 `OPENAI_API_KEY`). 키가 없으면 준비 셀이 그 자리에서 멈추고 무엇을 해야 하는지 알려 줍니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

In [ ]:
# [제공 코드] 데이터 파일 읽기: 이 셀은 실행만 하세요.
import json
from pathlib import Path

# 교안은 단원 폴더에서, 정답 노트북은 정답/ 폴더에서 돌아가므로 두 경로를 모두 본다.
_DATA = Path("data") if Path("data").exists() else Path("../data")


def load_jsonl(name):
    """data/<name> 을 한 줄씩 읽어 dict 리스트로 돌려준다(한 줄에 JSON 하나)."""
    rows = []
    for line in (_DATA / name).read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def load_json(name):
    """data/<name> 을 통째로 읽어 dict 로 돌려준다(사전 파일용)."""
    return json.loads((_DATA / name).read_text(encoding="utf-8"))


print("데이터 폴더:", _DATA)

---
# 1. LLM 으로 개체 뽑기

규칙 기반이 막혔던 바로 그 발췌를, 이번엔 모델에게 말로 시켜 봅니다. 소단원 하나로 끝나는 짧은 대단원입니다.

## 1-1. zero-shot 프롬프트로 지시하기

### 왜 될까요?
LLM 은 이미 방대한 글로 언어를 폭넓게 익힌 상태입니다. 그래서 **"이런 유형을 뽑아 줘"라고 말**만 하면 바로 시작합니다. 예시를 하나도 주지 않고 지시만 하는 방식을 **zero-shot** 이라고 합니다. 지난 시간 규칙 기반이 막혔던 그 발췌를 그대로 시켜 봅니다.

In [ ]:
# 지난 시간 규칙 기반이 statins 도 Warfarin 도 놓쳤던 바로 그 발췌다.
papers = load_jsonl('core_papers.jsonl')
text = papers[0]['text']                     # 이 원문 하나를 이 노트북 내내 데모로 쓴다
print(papers[0]['doc_id'], '/', papers[0]['journal'], papers[0]['year'])

In [ ]:
# 원문이 어떤 글인지 앞부분만 훑어 둔다. 뒤에서 뽑은 개체가 여기 있는지 계속 되짚게 된다.
print(text[:300], '...')

In [ ]:
# zero-shot: 정답 예시를 하나도 주지 않고 '무엇을 뽑을지'만 지시한다.
# 유형 다섯 가지를 프롬프트에 못 박는 것이 핵심이다. 안 적으면 모델이 제 나름의 유형 이름을 만들어 온다.
from langchain_core.prompts import ChatPromptTemplate

# 원문 자리를 변수로 비워 둔 틀이다. 문서가 바뀌어도 틀은 그대로 쓴다
zero_prompt = ChatPromptTemplate.from_template(
    '다음 논문 원문에서 개체를 뽑아 유형과 함께 나열해 주세요.\n'
    '유형은 Compound, Gene, Disease, Symptom, PharmacologicClass 중에서 고르세요.\n'
    '원문: {원문}')

chain = zero_prompt | make_model()     # 틀에 값을 끼우고 모델로 보내는 사슬
answer = chain.invoke({'원문': text})   # 모델을 한 번 부른다(실제 호출이 나간다)
print(answer.text)

> 지난 시간 규칙 기반이 놓친 **statins·proton pump inhibitors·Warfarin** 을, 재학습도 사전도 없이 **유형 이름만 알려 주고** 잡았습니다(모델은 이 이름들을 `Statins`·`Proton pump inhibitors (PPIs)` 처럼 첫 글자를 대문자로 고쳐 적었습니다. 이 표기 차이가 4-2 에서 문제가 됩니다).
>
> 다만 출력의 **모양**이 지난 시간과 다릅니다. 지난 시간에는 `find` 로 **구간(start·end)** 을 붙였는데, LLM 은 글자 번호를 잘 못 세기 때문에 구간 대신 **개체 텍스트**만 돌려줍니다. 그래서 3-2 에서는 구간 대조 대신 **"그 이름이 원문에 나오는가"** 로 검증합니다.
>
> 그리고 답이 **줄글**이라, 이대로는 프로그램이 개체를 하나씩 꺼내 쓰기 어렵습니다.

### ✅ 바로 확인 퀴즈

**1.** LLM 기반 NER 이 규칙 기반보다 **유리한** 점은?

<details><summary>정답 보기</summary>

**재학습이나 사전 없이 프롬프트만으로** 새 도메인의 개체와 유형을 뽑을 수 있습니다. 사전에 없는 이름(미등록어)과 붙어 있는 낱말(경계)에서 막히던 규칙 기반의 한계를 넘습니다.

</details>

---
# 2. 구조화 출력으로 개체 받기

1절의 답은 줄글이라 프로그램이 바로 못 씁니다. **`with_structured_output`** 으로 정해진 구조로 받습니다. LangChain 단원에서 쓴 그 함수를 개체 추출에 그대로 적용합니다.

## 2-1. 스키마로 형식을 강제한다

### 문법: 개체 스키마와 추출기
- `class Entity(BaseModel)` 로 개체 하나의 **필드**를 선언합니다. `name`·`type` 에 **`other_type`** 과 **`confidence`** 를 더한 넷입니다.
- `type` 은 `str` 이 아니라 **`Literal['Compound', ...]`** 로 씁니다. 이러면 다섯 값이 JSON 스키마의 **enum** 으로 실려 나가, 모델이 `drug`·`gene/enzyme` 같은 **제 맘대로 유형 이름을 못 붙입니다.** 스키마로 막을 수 있는 것은 스키마로 막는 것이 원칙입니다.
- 개체는 여러 개이므로 `class EntityList(BaseModel)` 로 **목록**을 감쌉니다.
- 프롬프트는 **`ChatPromptTemplate`** 로 만듭니다. 원문 자리만 변수로 비워 두면 문서가 바뀌어도 같은 틀을 씁니다(LangChain 단원에서 쓴 그 방식입니다).
- `틀 | make_model().with_structured_output(EntityList)` 로 사슬을 만들면 모델이 **그 구조로만** 답합니다.

In [ ]:
# 개체 하나의 모양: 이름, 유형, 그리고 Other 일 때 쓰는 칸
from typing import Literal, Optional
from pydantic import BaseModel, Field

class Entity(BaseModel):
    # description 은 주석이 아니라 모델에게 전달되는 지시다. 여기에 규칙을 적으면 출력이 달라진다.
    name: str = Field(description='개체 이름(원문에 나온 그대로)')   # '그대로' 라고 적어야 뒤의 원문 등장 확인이 통한다
    # type 은 str 이 아니라 Literal 이다. 다섯 값이 JSON 스키마의 enum 으로 실려 나가 모델이 그 밖을 못 쓴다
    type: Literal['Compound', 'Gene', 'Disease', 'Symptom', 'PharmacologicClass', 'Other'] = Field(description='개체 유형')
    # type 이 Other 로 온 개체가 실제로 무엇인지 여기에 적힌다. 나머지 개체는 None 이다
    other_type: Optional[str] = Field(
        default=None,
        description="type 이 Other 일 때만 채운다. 그 개체가 실제로 무엇인지 영어 한 낱말로 (예: Year, Institution)")
    # 모델이 스스로 매기는 점수다. 확률이 아니라 자기 보고라는 점을 뒤에서 확인한다
    confidence: float = Field(
        description="이 개체와 유형이 맞다고 얼마나 확신하는지 0.0 에서 1.0 사이")

class EntityList(BaseModel):
    # 개체가 여러 개이므로 리스트 필드 하나로 감싼다. 모델은 이 껍데기째로 답한다.
    entities: list[Entity] = Field(description='원문에서 뽑은 개체 목록')

In [ ]:
# 구조화 출력으로 개체를 뽑는 함수: with_structured_output 이 EntityList 틀을 강제한다
# system 문장도 같은 다섯 가지를 말한다. 스키마의 Literal 이 강제하고, 이 문장은 왜 그 다섯인지를 모델에게 알려 준다.
EXTRACT_SYSTEM = '의학 논문 원문에서 개체를 뽑아라. 유형은 Compound, Gene, Disease, Symptom, PharmacologicClass 중에서 고르고, 다섯에 안 드는 개체는 Other 로 적는다. 개체 이름은 원문에 나온 표현을 그대로 쓴다.'


# 규칙은 system 에, 원문은 user 에 나눠 넣는 틀. 원문 자리만 변수로 비워 둔다
EXTRACT_PROMPT = ChatPromptTemplate.from_messages([
    ('system', EXTRACT_SYSTEM),
    ('user', '{원문}'),
])

def extract_entities(text):
    """원문 하나에서 뽑은 Entity 객체 리스트를 돌려준다(모델 호출 1회)."""
    # 틀 | 모델 로 사슬을 만든다. with_structured_output 이 EntityList 모양을 강제한다
    chain = EXTRACT_PROMPT | make_model().with_structured_output(EntityList)
    # 답이 스키마와 어긋나면 이 줄 앞의 파싱 단계에서 예외가 난다(형식이 틀린 채로 넘어오지는 않는다).
    # 실무 코드에서는 이 호출을 try 로 감싸고, 예외가 나면 빈 목록을 돌려준다.
    return chain.invoke({'원문': text}).entities           # 껍데기(EntityList) 를 벗겨 목록만 돌려준다

In [ ]:
ents = extract_entities(text)   # 모델 호출 1회
for ent in ents:
    print(ent.name, '->', ent.type)

In [ ]:
# 모델 답이라 실행마다 개수가 조금 다를 수 있다
print('총', len(ents), '개')

### 확신도는 어디서 쓸모가 있나

스키마에 `confidence` 를 넣었으니 모델이 개체마다 점수를 함께 돌려줍니다. 낮은 쪽부터 봅니다.

In [ ]:
# 확신도가 어떻게 분포하는지 센다. 값보다 폭이 중요하다
from collections import Counter

print(Counter(round(ent.confidence, 2) for ent in ents))

> **값이 전부 위쪽에 몰려 있습니다.** 이 발췌는 의학 논문이라 개체가 명확해서 모델이 헷갈릴 자리가 없습니다. 폭이 이렇게 좁으면 "0.9 미만을 버린다" 같은 **임계값 필터가 아무것도 걸러 내지 못합니다.** 실행할 때마다 값이 조금씩 달라지지만 몰려 있는 모양은 같습니다.

값이 갈리는 자리를 일부러 만들어 봅니다. 지난 시간 규칙 기반이 **약어 충돌**로 막혔던 그 종류의 문장입니다.

In [ ]:
# PSD, OTC, SET, LARGE, CAT, ACT, CLOCK 은 유전자 기호이면서 흔한 약어이기도 하다
hard_text = ('Patients with PSD were given OTC drugs. The SET of markers, including LARGE '
             'and CAT, was measured after ACT therapy in the CLOCK study.')

for ent in sorted(extract_entities(hard_text), key=lambda e: e.confidence):   # 모델 호출 1회
    print(f'{ent.confidence:.2f}  {ent.name} ({ent.type})')

> **이번엔 갈립니다.** 약어처럼 여러 뜻으로 읽히는 자리에서 점수가 내려갑니다. 교안_01 3-5 에서 규칙 기반이 문맥을 못 읽어 손도 못 대던 바로 그 자리를, LLM 은 **"덜 확신한다"고 표시**할 수 있습니다.
>
> 다만 이 값의 성격을 분명히 해야 합니다. **교안_01 4-2 에서 본 BERT 의 `score` 는 모델이 계산한 확률**이지만, 여기 `confidence` 는 **모델이 스스로 매긴 자기 보고**입니다. 같은 문장을 다시 넣으면 값이 조금씩 달라지고, 1.00 이라고 해서 맞다는 보장도 없습니다.
>
> 그래서 쓰는 법이 다릅니다. **임계값으로 자동으로 버리는 데 쓰지 말고, 사람이 먼저 볼 것을 고르는 데 쓰세요.** 수백 편을 뽑아 놓고 낮은 순으로 정렬해 위에서부터 검수하면 같은 시간에 더 많은 오류를 잡습니다. 자동으로 거르는 일은 다음 3절의 후처리가 맡습니다.

> 이제 각 개체가 `Entity` 객체라 `ent.name`·`ent.type` 로 바로 꺼내 씁니다. 펜스를 벗기거나 파싱할 필요가 없습니다. **답이 왔다면 그 모양은 스키마대로**입니다. 이게 프롬프트 부탁과의 결정적 차이입니다.
>
> 다만 **답이 아예 안 오는 경우까지 막아 주지는 않습니다.** 모델이 스키마와 어긋난 답을 보내면 그 답이 조용히 통과하는 것이 아니라 **파싱 단계에서 예외**가 납니다. 실무 코드에서는 이 호출을 `try` 로 감싸고, 예외가 나면 빈 목록을 돌려주거나 한 번 더 부릅니다.

> `type` 을 `str` 로 두면 모델이 `drug`·`gene/enzyme` 처럼 유형 이름을 지어냅니다. 그래프의 노드 레이블과 안 맞아 하나도 못 붙입니다. `Literal` 로 묶으면 목록 밖 값이 나오지 않습니다.
>
> 목록에 **`Other`** 를 함께 넣은 이유가 있습니다. 다섯뿐이면 모델은 어디에도 안 맞는 개체를 만났을 때 **다섯 중 하나를 억지로 뒤집어씌웁니다**(연도가 `Disease` 로 들어오는 식입니다). `Other` 가 있으면 그대로 표시하고, **3절의 유형 검증이 그것을 버립니다.**
>
> `Other` 로 온 개체가 **실제로 무엇인지**는 `other_type` 칸에 적힙니다(`Year`·`Institution` 처럼). 버리기 전에 이 값을 모아 보면 **우리 유형 목록에 무엇이 빠졌는지**가 보입니다. 유형을 새로 넣을지 정하는 근거가 됩니다.

### 🖐️ 함께 따라하기: 다른 논문에서 유전자만 골라내기

데모와 **다른 논문**으로 추출기를 한 번 더 돌려 봅니다. `followalong_papers.jsonl` 의 **두 번째 문서**(인덱스 1)를 읽어 `extract_entities` 로 개체를 뽑은 뒤, 유형이 **`Gene`** 인 개체의 **이름만** 리스트로 모아 개수와 함께 출력해 보세요.

- 모델을 **한 번** 부릅니다.
- **확인 기준**: 유전자 이름이 **다섯 개 이상** 담기고, 그중 `STMN1`·`TUBB` 처럼 원문에 그대로 있는 기호가 보이면 맞은 것입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) load_jsonl('followalong_papers.jsonl') 의 인덱스 1 문서를 paper 에 담는다
# 2) extract_entities(paper['text']) 로 개체를 뽑는다
# 3) type 이 'Gene' 인 개체의 name 만 리스트로 모아 개수와 함께 출력한다

### ✅ 바로 확인 퀴즈

**1.** `with_structured_output(EntityList)` 로 받으면 무엇이 보장되고 무엇이 보장되지 않나요?

<details><summary>정답 보기</summary>

출력 **형식(필드·타입)을 스키마로 강제**하므로, 답이 오면 펜스·필드 누락·군더더기 없이 정해진 구조로 받습니다. 그래서 `json.loads` 파싱이나 펜스 벗기기가 필요 없습니다. 다만 모델이 스키마에 맞지 않는 답을 보내는 경우까지 없어지지는 않아서, 그때는 **예외**로 드러납니다.

</details>

---
# 3. 후처리: 뽑은 개체를 믿을 수 있게 다듬기

모양이 맞아도 **내용**은 완벽하지 않습니다. **원문에 없는 개체**를 지어내거나(환각), **같은 개체를 두 번** 뽑거나, 우리 유형에 안 맞는 개체에 **엉뚱한 유형을 뒤집어씌우기도** 합니다. 의료 데이터에서 이건 그냥 잡음이 아니라 **안전 문제**입니다. 그래서 뽑은 뒤에는 항상 세 가지로 **거릅니다**.

- **유형 검증**: 우리가 정한 허용 유형 다섯에 없는 것은 버립니다. 2-1 의 스키마가 내보내는 **`Other`** 가 여기서 걸러집니다. 스키마 없이 뽑은 옛 결과나 다른 팀이 넘겨준 파일을 받을 때도 같은 검사가 필요합니다.- **원문 등장 확인**: 개체 이름이 실제 논문 원문에 **나오는지** 확인합니다(환각 제거).
- **중복 제거**: 같은 `(이름, 유형)` 은 하나만 남깁니다.

실습을 안정적으로 하기 위해, 후처리는 **미리 저장해 둔 추출 결과**(`data/ie_raw_entities.jsonl`)로 연습합니다. 일부러 결함을 섞어 두었습니다.

- **3-1** 유형 검증
- **3-2** 원문 등장 확인
- **3-3** 중복 제거

In [ ]:
# 후처리 연습용으로 미리 뽑아 둔 개체를 읽는다. 일부러 결함(허용 밖 유형, 환각, 중복)을 섞어 두었다.
raw_entities = load_jsonl('ie_raw_entities.jsonl')
doc_text = {paper['doc_id']: paper['text'] for paper in papers}   # 2단계에서 원문을 doc_id 로 찾으려고 미리 만든다
print('추출 결과:', len(raw_entities), '개')

In [ ]:
# 개체마다 name 과 type 에 더해 어느 논문에서 나왔는지(doc_id)가 붙어 있다. 2단계가 그 doc_id 로 원문을 찾는다.
for r in raw_entities:
    print(r)

## 3-1. 유형 검증

허용 유형은 이 단원의 5종입니다. 그 집합에 없는 유형(예: `Guideline`)을 버립니다. `CPIC` 는 원문에 분명히 있는 말이지만 우리 그래프에 그런 노드가 없으므로 쓸 수 없습니다.

> 이 파일은 **`Literal` 없이 뽑았다고 가정한** 결과입니다. 그래야 이 단계가 무엇을 걸러 내는지 보이기 때문입니다.
>
> 2-1 의 스키마로 뽑았다면 여기서 걸리는 것은 **`Other`** 입니다. 버리기 전에 `other_type` 값을 모아 보세요. `Year` 가 계속 나온다면 우리 유형 목록에 그것을 넣을지 판단할 근거가 됩니다.

In [ ]:
# 허용 유형은 우리가 정한 계약이다. 계약에 없는 유형이 오면 그 개체는 그래프에 넣을 자리가 없다.
allowed_types = {'Compound', 'Gene', 'Disease', 'Symptom', 'PharmacologicClass'}   # 앞 단원 그래프의 노드 레이블 5종
type_ok = [r for r in raw_entities if r['type'] in allowed_types]
print('유형 검증 후:', len(type_ok), '개')

In [ ]:
# 몇 건이 빠졌나보다 '무엇이' 빠졌나가 중요하다. 버려진 쪽을 직접 본다.
print('버려진 것:', [r for r in raw_entities if r['type'] not in allowed_types])

## 3-2. 원문 등장 확인

개체 이름이 그 논문 원문에 실제로 나오는지 봅니다. 원문에 없으면 모델이 지어낸 것으로 보고 버립니다.

In [ ]:
# 코퍼스 전체가 아니라 '그 개체가 나왔다고 적힌 논문'의 원문하고만 대조한다.
# 전체와 대조하면 다른 논문에 우연히 있는 이름이 통과해 환각을 놓친다.
in_text = [r for r in type_ok if r['name'] in doc_text[r['doc_id']]]   # 대소문자까지 원문 그대로여야 통과한다
print('원문 등장 확인 후:', len(in_text), '개')

In [ ]:
# 걸러진 쪽을 따로 모은다. 몇 건이 빠졌나보다 '무엇이' 빠졌나가 이 단계의 교훈이다.
removed = [r['name'] for r in type_ok if r['name'] not in doc_text[r['doc_id']]]
print('원문에 없어 제거된 개체:', removed)

> 이 필터는 **날이 서 있어 진짜 개체까지 벨 수 있습니다.** `name in text` 는 대소문자까지 원문 그대로여야 통과시키는데, 모델은 이름을 문장 첫 글자처럼 **대문자로 고쳐** 적는 버릇이 있습니다. 1-1 의 zero-shot 답으로 확인해 봅니다.

In [ ]:
# 1-1 zero-shot 답이 이름을 어떻게 적었는지 원문과 맞춰 본다(모델을 다시 부르지 않는다).
for name in ['Statins', 'Omeprazole', 'Simvastatin']:
    # 가운데 칸이 False 인데 오른쪽이 True 면, 진짜 개체인데 대문자 때문에 걸러진다는 뜻이다.
    print(f'{name:14s} 1-1 답에 있나 {name in answer.text} / 원문에 그대로 {name in text} / 소문자로는 {name.lower() in text}')

> 셋 다 **진짜 개체**인데 원문은 `statins`·`omeprazole`·`simvastatin` 이라 소문자입니다. **이번 실행의 1-1 답**은 셋 다 대문자로 적었으므로(첫 칸이 `True`), 그 답을 그대로 2단계에 넣었다면 세 개가 **환각으로 몰려 버려졌을** 것입니다.
>
> 본인 키로 다시 돌리면 첫 칸이 `False` 로 나올 수도 있습니다. 그건 그 실행에서 모델이 원문 표기를 지켰다는 뜻이고, 그러면 이 필터도 그 답을 버리지 않습니다. **표기가 실행마다 흔들린다는 것 자체가** 이 필터가 왜 위험한지를 말해 줍니다.
>
> 그래서 2-1 스키마의 `Field(description='개체 이름(원문에 나온 그대로)')` 가 장식이 아닙니다. 그 한 줄이 모델에게 **표기를 고치지 말라**고 지시하고, 그 덕에 이 검사가 진짜 개체를 버리지 않습니다. 교재가 돌렸을 때 이 발췌의 구조화 출력은 이 검사에서 **한 건도** 걸러지지 않았습니다.
>
> 필터를 쓸 때는 늘 **무엇이 걸리는지**만이 아니라 **무엇이 잘못 걸리는지**도 함께 봐야 합니다.

### ✅ 바로 확인 퀴즈

**1.** '원문 등장 확인' 후처리가 걸러 내는 문제는 무엇인가요?

<details><summary>정답 보기</summary>

모델이 **원문에 없는 개체를 지어내는 환각**을 걸러 냅니다. 개체 이름이 실제 논문에 나오는지 확인해, 나오지 않으면 버립니다. 의료 도메인에서는 이 한 단계가 "논문이 말한 것"과 "모델이 지어낸 것"을 가르는 경계입니다.

</details>

## 3-3. 중복 제거

같은 `(이름, 유형)` 쌍은 하나만 남깁니다. 이미 본 쌍을 `set` 에 기록하며 거릅니다.

In [ ]:
# 같은 (이름, 유형) 은 처음 만난 것 하나만 남긴다.
seen = set()
clean = []
for r in in_text:
    # 이름만이 아니라 유형까지 묶어 키로 쓴다. 같은 이름이 다른 유형으로 뽑혔다면 서로 다른 개체다.
    key = (r['name'], r['type'])
    if key not in seen:
        seen.add(key)
        clean.append(r)
print('중복 제거 후 최종:', len(clean), '개')

In [ ]:
# 세 단계를 다 지난 개체 목록이다. 이대로 그래프에 얹을 수 있는 상태다.
for r in clean:
    print(r['name'], '->', r['type'])

> 처음 **12개**가 유형 검증(→11) · 원문 등장 확인(→9) · 중복 제거(→**8**)를 거쳐 믿을 수 있는 개체만 남았습니다. 이 세 단계가 LLM 추출을 **실무에서 쓸 수 있게** 만드는 마무리입니다.
>
> 특히 3-2 가 걸러 낸 `Aspirin`·`BRCA1` 을 보세요. 둘 다 의학적으로 그럴듯한 이름이라 사람 눈으로는 잘 걸러지지 않습니다. **이 발췌가 말하지 않은 사실**을 그래프에 넣으면, 나중에 그 그래프로 답한 문장에는 근거가 없습니다.

<img src="images/postprocess_funnel.png" width="900">

### 🖐️ 함께 따라하기: 다른 파일에서 유형별 개수 세기

데모가 쓴 `ie_raw_entities.jsonl` 말고, **다른 논문 세 편**에서 뽑아 둔 `data/ie_followalong_entities.jsonl` 로 같은 집계를 해 보세요.

- `load_jsonl` 로 그 파일을 읽고, 유형별 개체 수를 `collections.Counter` 로 세어 출력합니다.
- **확인 기준**: 세어 보면 개체가 모두 **12개**이고 **`Gene`** 유형이 **10개**로 가장 많습니다. 두 숫자가 맞으면 제대로 센 것입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) load_jsonl('ie_followalong_entities.jsonl') 로 개체를 읽는다
# 2) collections 에서 Counter 를 가져온다
# 3) 각 개체 type 을 세어 출력하고, 전체 개수도 함께 출력한다

---
# 4. 배치 적용과 규칙 대 LLM 비교

문서 하나가 아니라 문서 더미로 넓히고, 지난 시간의 규칙 기반과 나란히 놓아 무엇이 갈리는지 봅니다.

- **4-1** 배치 추출과 유형 분포
- **4-2** 규칙 기반과 LLM 나란히 보기
- **4-3** few-shot 은 형식을 유도할 뿐 (프롬프트 기법 되짚기)

> **4-3 이 왜 여기 있나요?** 1-1 의 zero-shot 과 짝이 되는 few-shot 을 일부러 **맨 뒤로** 미뤘습니다. 2절에서 구조화 출력으로 형식을 강제해 본 뒤에야 "프롬프트로 형식을 유도하는 것"이 어디까지인지 견줄 기준이 생기기 때문입니다.

## 4-1. 배치 추출과 유형 분포

### 왜 필요할까요?
실무에선 논문 하나가 아니라 **문서 더미 전체**에서 개체를 뽑습니다. 추출기를 **반복문으로** 모든 논문에 적용하고, 뽑힌 개체의 **유형 분포**를 집계해 봅니다. 그리고 지난 시간의 **규칙 기반**과 나란히 비교합니다.

> **호출 수 주의**: 아래 셀은 논문마다 한 번씩, 코퍼스 6편이므로 **모두 6회** `.invoke` 를 부릅니다. 그중 첫 논문은 2-1 에서 이미 같은 프롬프트로 불렀으니, 그 답을 변수에 담아 두었다면 다섯 번만 불러도 됩니다. **같은 프롬프트를 다시 부르지 않는 것**도 비용을 줄이는 방법입니다.
>
> 여기서는 6편이지만 실무 코퍼스는 수천 편이라, 문서 수만큼 비용과 시간이 곧바로 늘어납니다. 그래서 상한을 걸거나 규칙 기반으로 1차를 거른 뒤 LLM 을 쓰기도 합니다.

In [ ]:
# 코퍼스의 모든 논문에 추출기를 적용해 개체를 모은다(논문 6편 = 모델 호출 6회).
all_entities = []
for paper in papers:
    for ent in extract_entities(paper['text']):   # 논문 하나에서 여러 개체가 나오므로 이중 반복이다
        all_entities.append(ent)

from collections import Counter
print('전체 개체 수:', len(all_entities))
print('유형 분포:', Counter(ent.type for ent in all_entities))

> 여러 논문을 한 번에 처리해 유형 분포를 봤습니다(모델 출력이라 실행마다 숫자는 조금 다를 수 있습니다). 이제 **같은 발췌** 하나에 규칙 기반과 LLM 을 나란히 적용해 차이를 눈으로 봅니다.

## 4-2. 규칙 기반과 LLM 나란히 보기

지난 시간에 만든 규칙 기반 NER 을 그대로 다시 제공합니다. 같은 발췌에 두 방식을 나란히 걸어 무엇이 갈리는지 봅니다.

In [ ]:
# [제공 코드] 지난 시간에 만든 규칙 기반 NER: 비교용으로 다시 제공합니다(실행만 하세요).
import re

dictionary = load_json("name2id.json")
entries = dictionary["entries"]      # 소문자로 통일한 이름 -> {id, label, canonical, source}
genes = dictionary["genes"]          # 유전자 기호 -> id (대소문자 그대로)
# 평범한 영어 낱말과 철자가 같은 이름들. 소문자로 쓰인 자리는 개체로 보지 않는다.
brand_stopwords = set(dictionary["brand_stopwords"])

TOKEN = re.compile(r"[A-Za-z][A-Za-z0-9\-]*")   # 숫자와 하이픈을 낱말의 일부로 본다(CYP2C19, HLA-A)


def rule_based_ner(text):
    """사전에 있는 낱말을 (구간, 유형)이 붙은 개체 dict 리스트로 돌려준다."""
    found = []
    for match in TOKEN.finditer(text):
        word = match.group()
        if word in genes:                       # 유전자 기호는 대소문자가 뜻이라 그대로 맞춘다
            entity_type = "Gene"
        elif word.lower() in entries:
            if word.lower() in brand_stopwords and word.islower():
                continue                        # 상품명이 소문자로 쓰였으면 평범한 낱말이다
            entity_type = entries[word.lower()]["label"]
        else:
            continue                            # 사전에 없는 낱말은 여기서 통째로 빠진다(미등록어 한계)
        found.append({"name": word, "type": entity_type,
                      "start": match.start(), "end": match.end()})
    return found


print("규칙 기반 NER 준비 완료: rule_based_ner(원문)")

In [ ]:
# 같은 발췌를 두 방식에 각각 넣어 무엇을 잡는지 나란히 본다(이름이 여러 번 나오므로 집합으로 비교한다).
rule_names = {ent['name'] for ent in rule_based_ner(text)}   # dict 라 대괄호로 꺼낸다
llm_names = {ent.name for ent in extract_entities(text)}     # 객체라 점으로 꺼낸다
print('규칙 기반이 잡은 이름:', len(rule_names), '개')
print('LLM 이 잡은 이름   :', len(llm_names), '개')

In [ ]:
# 개수보다 '어느 쪽만 잡았나'가 중요하다. 집합끼리 빼면 그 차이가 그대로 남는다.
print('규칙만 잡은 것:', sorted(rule_names - llm_names))
print('LLM 만 잡은 것 :', sorted(llm_names - rule_names))

> 규칙 기반이 잡은 것은 LLM 도 전부 잡았고, LLM 은 그 위에 몇 개를 더 잡았습니다. 그런데 **LLM 만 잡은 그 이름들**이 새로운 문제를 만듭니다. 사전에 있는지 확인해 봅시다.

In [ ]:
# LLM 이 잡은 이름을 사전에서 찾아 본다. 사전에 있어야 그래프의 어느 노드인지 정할 수 있다.
unknown = [name for name in sorted(llm_names)
           if name not in genes and name.lower() not in entries]   # 유전자 칸에도 이름 칸에도 없는 것만 남긴다
print('사전에 없는 이름:', unknown)

In [ ]:
# 이 이름들이 전부 원문에 있다면 환각이 아니라 사전이 모자란 것이다.
for name in unknown:
    print(f'{name:24s} 원문에 있나 {name in text}')

> 여기가 이 단원의 **다음 문제**입니다. `statins`·`proton pump inhibitors`·`PPIs`·`NSAIDs` 는 원문에 분명히 있고 유형도 맞지만, **사전에 없어 그래프의 어느 노드인지 정할 수 없습니다**.
>
> 이 이름들을 그대로 그래프에 넣으면 어떻게 될까요. `PPIs` 와 `proton pump inhibitors` 가 **서로 다른 노드**가 되고, 기존 그래프의 어떤 노드와도 이어지지 않은 **외딴 점**이 됩니다. 그래서 이 단원 뒤에서 두 가지를 배웁니다.
>
> 이 이름들을 어떻게 다룰지는 이 단원 뒤에서 이어집니다(**정규화**와 **격리 적재**).
>
> 규칙 기반과 LLM 은 경쟁이 아니라 **역할 분담**입니다. 규칙 기반은 빠르고 비용이 없으며 **id 를 바로** 얻습니다. LLM 은 사전 밖을 잡지만 **id 가 없는 이름**을 함께 데려옵니다.

<img src="images/rule_vs_llm.png" width="900">

> 그림의 **25** 는 교재가 한 번 돌려 잰 값입니다. 여러분 화면에서는 다를 수 있어요. **20** 쪽은 사전을 훑는 규칙이라 누가 돌려도 같습니다. 그 비대칭 자체가 이 그림의 요점이기도 합니다.

오늘 실행으로 확인한 것만 표로 정리하면 이렇습니다. 어느 쪽이 더 좋은지가 아니라, **어디에 무엇을 쓸지**를 정하는 표입니다.

| 재는 축 | 규칙 기반 | LLM |
|---|---|---|
| 사전 밖 이름(`statins`·`PPIs`) | 못 잡는다 | 잡는다 |
| 그래프 노드 id | 사전에서 **바로** 나온다 | 없다(정규화가 따로 필요) |
| 호출 비용 | 없다 | 문서 수만큼 든다 |
| 실행마다 흔들리나 | 안 흔들린다 | 흔들린다(표기·유형 배정) |

### ✅ 바로 확인 퀴즈

**1.** 대량·정형 텍스트에는 규칙 기반이, 새 표현이 계속 나오는 논문에는 LLM 이 강한 이유를 한 문장으로 말해 보세요.

<details><summary>정답 보기</summary>

규칙 기반은 **빠르고 일관되며 사전의 id 를 바로** 얻지만 사전 밖을 못 잡고, LLM 은 **재학습 없이 문맥을 이해**해 새 표현을 잡지만 비용·지연·흔들림이 있고 **id 없는 이름**을 데려오기 때문입니다. 그래서 역할을 나눠 씁니다.

</details>

## 4-3. few-shot 은 형식을 유도할 뿐

### 왜 해 볼까요?
프롬프트로 형식을 다루는 마지막 수단은 **예시**입니다. 지시로도 강제로도 아니고, 답의 본보기를 하나 보여 주는 방식입니다. 정말 형식이 따라오는지, 대신 무엇이 흔들리는지 직접 재 봅니다.

### 🖐️ 함께 따라하기: 예시를 붙여(few-shot) 결과 비교

정답 예시를 몇 개 붙여 주는 방식을 **few-shot** 이라고 합니다. `followalong_papers.jsonl` 의 **네 번째 문서**(인덱스 3)로, 예시 없이(zero-shot) 한 번 · 예시 하나를 붙여(few-shot) 한 번, 모두 **2회** 부르고 두 답을 비교해 보세요.

두 프롬프트는 아래 문구를 **그대로** 쓰세요. **`ChatPromptTemplate.from_template`** 로 만들고, 원문 자리는 `{원문}` 변수로 비워 둡니다.

```text
다음 논문 원문에서 개체를 뽑아 유형과 함께 나열해 주세요.
유형은 Compound, Gene, Disease, Symptom, PharmacologicClass 중에서 고르세요.
원문: {원문}
```

```text
예시)
원문: Carbidopa, used to treat Parkinson disease, was reported to activate AHR.
개체: Carbidopa(Compound), Parkinson disease(Disease), AHR(Gene)

위 예시처럼 유형을 붙여 개체를 뽑아 주세요.
유형은 Compound, Gene, Disease, Symptom, PharmacologicClass 중에서 고르세요.
원문: {원문}
```

> **문구를 바꾸면 답도 바뀝니다.** 여기서 하는 실험이 그것이라 마음껏 고쳐 보되, 아래 확인 기준은 원래 문구로 돌린 뒤에 보세요.

- **확인 기준 1**: 두 답 모두 `HSD11B1`·`PTGS2` 를 잡으면 기본은 맞은 것입니다.
- **확인 기준 2**: 두 답의 **생김새**를 나란히 보세요. zero-shot 은 `### Compound` 같은 **소제목으로 나눈 문서**로 오고, few-shot 은 예시에 쓴 `개체:` 라벨을 그대로 받아 옵니다. **예시 한 개로도 형식은 예시 쪽으로 기웁니다.**
- **확인 기준 3**: few-shot 답에서 **같은 이름이 두 유형에 걸쳐 있는 것이 몇 개인지 세어 보세요.** **0 이면** 이번 실행은 형식만 기운 것이고, **1 이상이면** 형식이 좋아지는 동안 유형 배정이 흔들린 것입니다. 교재가 돌렸을 때는 `HSD11B1`·`PTGS2` 같은 유전자가 `Gene` 과 `Compound` 양쪽에 올라왔습니다. 어느 쪽이 나오든, **한 가지를 좋게 만들면 다른 것이 나빠질 수 있다**는 것을 세어서 확인하는 것이 이 절의 목적입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) load_jsonl('followalong_papers.jsonl') 의 인덱스 3 문서를 읽어 원문을 batch_text 에 담는다
# 2) 예시 없이 지시만 하는 zero_prompt 틀을 만들고 (틀 | 모델) 로 한 번 불러 출력한다
# 3) 예시 한 쌍을 앞에 붙인 few_prompt 틀로 한 번 더 부르고 출력한다
# 4) 두 출력의 개체와 형식을 비교한다

> **형식은 예시를 따라왔고, 유형 배정은 나빠졌습니다.** 교재가 이 문서로 돌렸을 때 zero-shot 은 유형마다 소제목을 달았고, few-shot 은 예시의 `개체:` 라벨을 받아 한 줄로 묶었습니다. 그런데 같은 유전자가 `Gene` 과 `Compound` **양쪽에** 올라왔고, 식물 이름이 `Compound` 로 들어왔습니다. **개체 수만 세면 이 악화가 안 보입니다.**
>
> 한 문서·한 프롬프트·한 모델에서 나온 **관찰**입니다. 예시를 더 넣거나 모델을 바꾸면 달라집니다. 남는 결론은 하나입니다.
>
> > **프롬프트로 형식을 유도할 수는 있어도 보장할 수는 없습니다.** 보장이 필요하면 **2-1 의 구조화 출력**을 씁니다.

> 그리고 지금까지의 비교는 전부 **개수**였습니다. 규칙 기반 20개, LLM 25개. 규칙 쪽 20은 사전을 훑는 것이라 **누가 돌려도 같고**, LLM 쪽 25는 **교재가 한 번 잰 값**이라 여러분 화면에서는 몇 개 달라질 수 있습니다. 어느 쪽이든 개수는 "얼마나 많이 잡았나"이지 **"얼마나 맞게 잡았나"가 아닙니다.** 어느 쪽이 더 정확한지 재려면 사람이 만든 **정답 라벨**과 견줘야 하고, 그 방법은 이 단원 뒤에서 배웁니다.

### ✅ 바로 확인 퀴즈 (4-2·4-3)

**1.** 규칙 기반이 20개, LLM 이 25개를 잡았습니다. 이것만 보고 "LLM 이 더 정확하다"고 말할 수 있을까요?

<details><summary>정답 보기</summary>

없습니다. 개수는 **얼마나 많이 잡았나**일 뿐, 그중 몇 개가 **맞는 개체인지**는 말해 주지 않습니다. 더 잡은 다섯 개가 진짜 개체일 수도 있고 잘못된 유형이 붙은 것일 수도 있습니다. few-shot 따라하기에서 본 것처럼 **개체 수가 늘어도 늘어난 것이 오분류와 중복일** 수 있습니다. 정확도를 재려면 사람이 만든 **정답 라벨**과 견줘야 하고, 그 방법은 이 단원 뒤에서 배웁니다.

</details>

---
## 🚀 응용 클론코딩: `extract_and_clean()` 로 묶기

원문 하나를 받아 **추출 → 유형 검증 → 원문 등장 확인 → 중복 제거**까지 한 번에 하는 `extract_and_clean(text)` 를 완성해 보세요. 반환은 정제된 개체 리스트(`{'name','type'}` dict)입니다.

- 데모 원문 `text` 에 걸어 **정제 후 개수**를 찍고, 2-1 의 `len(ents)` 와 견줘 보세요.
- 걸러진 개체가 있다면 그 목록도 함께 보세요.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) extract_entities(text) 로 개체를 뽑는다(Entity 객체 리스트)
# 2) 유형이 allowed_types 에 있고, 이름이 text 안에 있는 것만 남긴다
# 3) (이름, 유형) 중복을 제거해 dict 리스트로 돌려준다
# 4) 데모 원문 text 에 걸어 정제 후 개수와 목록을 출력한다

> **교재가 돌렸을 때는 걸러진 것이 0건이었습니다.** 정제 전과 후의 개수가 같았어요. 허용 밖 유형도, 원문에 없는 이름도, 중복도 없었다는 뜻입니다. **여러분 화면에서는 몇 건이 걸러질 수도 있습니다.** 그때는 어느 검사에 걸렸는지 보세요. 그것이 이 절에서 볼 것입니다.
>
> 4절에서 12개가 8개로 줄어든 것은 우리가 **결함을 일부러 심어 둔 파일**(`ie_raw_entities.jsonl`)이었기 때문입니다. 실제 출력은 이번처럼 깨끗할 때가 많습니다.
>
> 그래도 필터를 뺄 수는 없습니다. 이 세 가지는 **늘 생기는 일**이 아니라 **생겼을 때 그래프를 망치는 일**입니다. 논문 수백 편을 돌리면 그중 몇 편에서 반드시 나오고, 그때는 이미 잘못된 노드가 그래프에 들어간 뒤입니다. **비용이 거의 없는 검사로 드물지만 치명적인 사고를 막는 것**이 후처리의 자리입니다.

---
## 참고: `with_structured_output` 이 형식을 잡는 세 가지 방법

`with_structured_output` 은 `method` 인자로 방식을 고를 수 있습니다. 오늘 우리가 쓴 것은 **기본값**입니다.

| method | 어떻게 잡나 | 비고 |
|---|---|---|
| `json_schema` | 스키마를 **모델 요청에 직접 실어** 보내고 답을 그 스키마로 파싱 | **기본값**. 오늘 쓴 것 |
| `json_mode` | `json_schema` 의 옛 이름 | 같은 동작으로 넘어간다 |
| `function_calling` | 스키마를 **도구(함수) 정의**로 바꿔 도구 호출로 받기 | 예전 기본값. 지금은 권장되지 않는다 |

> `langchain-google-genai` 와 `langchain-openai` 둘 다 기본값이 `json_schema` 입니다. 예전에는 `function_calling` 이 기본이었고, 그 시절 코드는 "모델이 도구 호출을 못 하면 `None` 이 온다"를 방어했습니다. **지금 기본 경로에서는 `None` 이 오지 않습니다.** 어긋난 답은 파싱 단계에서 **예외**로 드러납니다. 오래된 예제 코드를 옮겨 쓸 때 이 차이를 확인하세요.

---
## 이번 강의 정리

| 단계 | 하는 일 | 핵심 |
|---|---|---|
| LLM 추출 | 프롬프트로 개체 뽑기 | `ChatPromptTemplate`, zero-shot·few-shot |
| 구조화 출력 | 형식과 유형을 강제 | `with_structured_output(EntityList)`, `Literal` + `Other` |
| 후처리 | 믿을 수 있게 다듬기 | 유형 검증·원문 등장·중복 제거 (12→11→9→8) |
| 배치·비교 | 여러 논문 + 규칙 대 LLM | 반복문·`Counter`·사전에 없는 이름 |

- LLM 은 재학습 없이 프롬프트로 개체를 뽑아 규칙 기반의 미등록어·경계 한계를 넘습니다.
- 줄글 답은 프로그램이 못 쓰므로 **`with_structured_output`** 으로 형식을 강제하고, `type` 은 **`Literal`** 로 값까지 묶습니다.
- 뽑은 개체는 **유형 검증·원문 등장 확인·중복 제거**로 다듬어야 실무에서 씁니다.
- 대신 LLM 은 **사전에 없는 이름**을 데려옵니다. 그래프에 넣으려면 이름이 아니라 **id** 가 필요합니다.
- **few-shot 은 형식을 유도할 뿐 보장하지 않습니다.** 이번 문서에서는 형식이 예시를 따라온 대신 유형 배정이 오히려 어긋났습니다. 도움이 되는지는 문서·프롬프트·모델마다 달라 **재 봐야** 압니다.
- 지금까지의 비교는 전부 **개수**였습니다. 어느 쪽이 더 **정확한지**는 정답 라벨과 견줘야 알 수 있고, 그 방법은 이 단원 뒤에서 배웁니다.
- 비용은 **호출 수**로 정해집니다. 논문마다 한 번이면 문서 수가 곧 호출 수이니, 코퍼스를 늘리기 전에 상한을 먼저 정합니다.

## ⏭️ 예고: 다음 시간

오늘은 개체를 뽑았습니다. 다음 걸음은 그 개체들을 **잇는 일**입니다. "이 약이 이 유전자에 결합한다"를 뽑아 개체 사이를 연결하고, 오늘 익힌 구조화 출력을 그 연결에도 적용합니다. 그러면서 한 가지를 더 정합니다. 어떤 관계 타입만 쓸지 **미리 계약으로 정하는 일**, 곧 온톨로지입니다.

수고하셨습니다!